# BoatFinder

Set a start date and end date, build the list of CMS dock video URLs, then download/process/delete one video at a time.

In [2]:
# Colab setup: create the helper files directly in the runtime.
# This avoids the files.upload() widget, which can hang in some Colab/IDE sessions.
from pathlib import Path

Path("getvideourls.py").write_text(r'''
from datetime import date, datetime, timedelta
from urllib.parse import urljoin
import re

import requests


DEFAULT_BASE_URL = "https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/"


def _parse_date(value):
    if isinstance(value, date):
        return value

    return datetime.strptime(value, "%Y-%m-%d").date()


def _date_range(start_date, end_date):
    current_date = _parse_date(start_date)
    final_date = _parse_date(end_date)

    if current_date > final_date:
        raise ValueError("start_date must be before or equal to end_date")

    while current_date <= final_date:
        yield current_date
        current_date += timedelta(days=1)


def get_video_urls(start_date, end_date, base_url=DEFAULT_BASE_URL):
    base_url = base_url.rstrip("/") + "/"
    video_urls = []

    for current_date in _date_range(start_date, end_date):
        day_url = urljoin(
            base_url,
            f"{current_date:%Y}/{current_date:%m}/{current_date:%d}/",
        )

        response = requests.get(day_url, timeout=30)
        response.raise_for_status()

        filenames = re.findall(r"cms.*?\.mp4", response.text)

        for filename in filenames:
            video_urls.append(urljoin(day_url, filename))

    return video_urls
''')

Path("videodownload.py").write_text(r'''
from pathlib import Path

import requests


def download_video(video_url, output_folder="/content"):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    filename = video_url.rstrip("/").split("/")[-1]
    local_path = output_folder / filename

    with requests.get(video_url, stream=True, timeout=30) as response:
        response.raise_for_status()

        with open(local_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    return str(local_path)
''')

from getvideourls import get_video_urls
from videodownload import download_video

print("Helper files created and imported.")

Helper files created and imported.


In [3]:
from pathlib import Path
import os

# get_video_urls and download_video are imported in the Colab setup cell above.

In [4]:
# Inputs: change only these dates for a new run.
start_date = "2022-10-01"
end_date = "2022-10-17"

# In Colab, /content is temporary storage and is a good place for one video at a time.
temporary_video_folder = "/content"

In [5]:
video_urls = get_video_urls(start_date, end_date)

print(f"Found {len(video_urls)} videos")
video_urls[:5]

Found 2410 videos


['https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-110405Z.mp4',
 'https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-110405Z.mp4',
 'https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-111403Z.mp4',
 'https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-111403Z.mp4',
 'https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-112404Z.mp4']

In [6]:
for num, video_url in enumerate(video_urls):
    print(f"Processing video {num + 1} of {len(video_urls)}")
    print(video_url)

    video_path = download_video(video_url, output_folder=temporary_video_folder)
    print(f"Downloaded to: {video_path}")

    try:
        print(f"num = {num}")

        # video_path is the variable you will pass into pythontester/process_video.
        # Example later:
        # crossing_events = process_video(video_path)
        pass

    finally:
        if os.path.exists(video_path):
            os.remove(video_path)
            print(f"Deleted: {video_path}")

Processing video 1 of 2410
https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-110405Z.mp4
Downloaded to: /content/cms_dock_south-2022-10-01-110405Z.mp4
num = 0
Deleted: /content/cms_dock_south-2022-10-01-110405Z.mp4
Processing video 2 of 2410
https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-110405Z.mp4
Downloaded to: /content/cms_dock_south-2022-10-01-110405Z.mp4
num = 1
Deleted: /content/cms_dock_south-2022-10-01-110405Z.mp4
Processing video 3 of 2410
https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-111403Z.mp4
Downloaded to: /content/cms_dock_south-2022-10-01-111403Z.mp4
num = 2
Deleted: /content/cms_dock_south-2022-10-01-111403Z.mp4
Processing video 4 of 2410
https://stage-ams.srv.axds.co/archive/mp4/uncw/cms_dock_south/2022/10/01/cms_dock_south-2022-10-01-111403Z.mp4
Downloaded to: /content/cms_dock_south-2022-10-01-111403Z.mp4
num = 3


KeyboardInterrupt: 